# Batch Operations — Bulk Ingestion and Export

When warming a cold cache or migrating data, issuing one `store()` per entry is
slow: each call triggers a separate embedding round-trip. Medha provides several
bulk APIs that batch-embed multiple questions in a single call:

- **`store_batch(entries)`** — simple list of dicts, one embedding call
- **`store_many(entries, batch_size, on_progress, ttl)`** — chunked, optional concurrent embedding, progress callback
- **`warm_from_file(path)`** — load from JSON array or JSONL file
- **`warm_from_dataframe(df)`** — load from a `pandas.DataFrame`
- **`export_to_dataframe()`** — export entire collection to a DataFrame
- **`dedup_collection()`** — remove duplicate entries sharing the same query hash

**Requirements:** `pip install "medha-archai[fastembed]"`  
Cells 6–7 also require `pip install pandas`.

In [ ]:
from medha import Medha, Settings
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

## Cell 1: `store_batch()` — Single Embedding Round-Trip

`store_batch()` accepts a flat list of dicts and embeds all questions in a
single `aembed_batch()` call before upserting them. Use it when you have a
moderate number of entries (up to a few hundred) and don't need TTL control
or a progress callback.

In [ ]:
settings = Settings(backend_type="memory")

entries = [
    {"question": "How many users?",      "generated_query": "SELECT COUNT(*) FROM users"},
    {"question": "List all products",    "generated_query": "SELECT * FROM products"},
    {"question": "Revenue last month",   "generated_query": "SELECT SUM(amount) FROM orders WHERE month = LAST_MONTH()"},
    {"question": "Active sessions",      "generated_query": "SELECT COUNT(*) FROM sessions WHERE active = true"},
    {"question": "Top 10 customers",     "generated_query": "SELECT * FROM customers ORDER BY spend DESC LIMIT 10"},
]

async with Medha("batch_basic", embedder=embedder, settings=settings) as medha:
    ok = await medha.store_batch(entries)
    print(f"store_batch succeeded: {ok}")

    count = await medha._backend.count("batch_basic")
    print(f"Entries in backend  : {count}")

    hit = await medha.search("How many users are there?")
    print(f"Search hit strategy : {hit.strategy.value}")

## Cell 2: `store_many()` with `on_progress` Callback

`store_many()` chunks the input into `batch_size` slices and calls
`on_progress(done, total)` after each chunk. Use it for large ingestion
jobs where you want a progress bar.

In [ ]:
settings = Settings(backend_type="memory")

# 50 synthetic entries
entries = [
    {"question": f"Query about topic {i}", "generated_query": f"SELECT * FROM topic_{i}"}
    for i in range(50)
]

progress_log: list[tuple[int, int]] = []

def on_progress(done: int, total: int) -> None:
    progress_log.append((done, total))
    pct = done / total * 100
    print(f"  Progress: {done}/{total} ({pct:.0f}%)")

async with Medha("batch_progress", embedder=embedder, settings=settings) as medha:
    stored = await medha.store_many(entries, batch_size=10, on_progress=on_progress)
    print(f"\nStored {stored} entries in {len(progress_log)} chunks")

## Cell 3: `store_many()` with Concurrent Embedding

`Settings(batch_embed_concurrency=N)` tells `store_many()` to embed `N`
chunks concurrently via `asyncio.gather`. This is beneficial when the embedder
is remote (e.g. OpenAI or a GPU server) and can handle parallel requests.

In [ ]:
import time

entries = [
    {"question": f"Concurrent entry {i}", "generated_query": f"SELECT {i}"}
    for i in range(40)
]

# Sequential embedding (default)
settings_seq = Settings(backend_type="memory", batch_embed_concurrency=1)
async with Medha("batch_seq", embedder=embedder, settings=settings_seq) as medha:
    t0 = time.monotonic()
    await medha.store_many(entries, batch_size=10)
    t_seq = (time.monotonic() - t0) * 1000

# Concurrent embedding (2 chunks at a time)
settings_con = Settings(backend_type="memory", batch_embed_concurrency=2)
async with Medha("batch_con", embedder=embedder, settings=settings_con) as medha:
    t0 = time.monotonic()
    await medha.store_many(entries, batch_size=10)
    t_con = (time.monotonic() - t0) * 1000

print(f"Sequential  : {t_seq:.0f} ms")
print(f"Concurrent  : {t_con:.0f} ms")
print("(Speed difference is visible with remote embedders, less so with local FastEmbed)")

## Cell 4: `warm_from_file()` — JSON Array

`warm_from_file()` reads a file and calls `store_many()` internally.
It supports two formats:
- **JSON array**: `[{"question": ..., "generated_query": ...}, ...]`
- **JSONL**: one JSON object per line

Optional per-entry keys: `response_summary`, `template_id`.

In [ ]:
import json
import tempfile
import os

data = [
    {"question": "Total orders",        "generated_query": "SELECT COUNT(*) FROM orders"},
    {"question": "Pending shipments",   "generated_query": "SELECT * FROM shipments WHERE status = 'pending'"},
    {"question": "Average order value", "generated_query": "SELECT AVG(amount) FROM orders",
     "response_summary": "Average order value across all time"},
]

# Write a temporary JSON file
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, encoding="utf-8") as f:
    json.dump(data, f)
    json_path = f.name

settings = Settings(backend_type="memory")

async with Medha("batch_json", embedder=embedder, settings=settings) as medha:
    stored = await medha.warm_from_file(json_path)
    print(f"Loaded {stored} entries from JSON array")

    hit = await medha.search("How many orders are there?")
    print(f"Search result: {hit.strategy.value} — {hit.generated_query}")

os.unlink(json_path)

## Cell 5: `warm_from_file()` — JSONL

JSONL (one JSON object per line) is memory-efficient for very large datasets
because it streams line-by-line without loading the whole file into RAM.

In [ ]:
import json
import tempfile
import os

lines = [
    {"question": "Monthly active users",   "generated_query": "SELECT COUNT(DISTINCT user_id) FROM events WHERE DATE_TRUNC('month', ts) = DATE_TRUNC('month', NOW())"},
    {"question": "Churn rate this quarter", "generated_query": "SELECT churn_rate FROM metrics WHERE period = CURRENT_QUARTER()"},
    {"question": "Top 5 countries by GMV",  "generated_query": "SELECT country, SUM(gmv) AS total FROM orders GROUP BY country ORDER BY total DESC LIMIT 5"},
]

with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False, encoding="utf-8") as f:
    for item in lines:
        f.write(json.dumps(item) + "\n")
    jsonl_path = f.name

settings = Settings(backend_type="memory")

async with Medha("batch_jsonl", embedder=embedder, settings=settings) as medha:
    stored = await medha.warm_from_file(jsonl_path)
    print(f"Loaded {stored} entries from JSONL")

    hit = await medha.search("Users active this month")
    print(f"Search result: {hit.strategy.value} — {hit.generated_query[:60]}...")

os.unlink(jsonl_path)

## Cell 6: `warm_from_dataframe()` — Load from a pandas DataFrame

If your question/query pairs live in a DataFrame (e.g. exported from a
database or a BI tool), `warm_from_dataframe()` maps columns by name.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "question": [
        "Daily revenue",
        "Weekly signups",
        "Conversion rate",
        "Cart abandonment rate",
    ],
    "generated_query": [
        "SELECT SUM(amount) FROM orders WHERE DATE(created_at) = CURDATE()",
        "SELECT COUNT(*) FROM users WHERE YEARWEEK(created_at) = YEARWEEK(NOW())",
        "SELECT converted / visited AS rate FROM funnel",
        "SELECT abandoned / initiated AS rate FROM carts",
    ],
    "response_summary": [
        "Today's total revenue",
        "Signups this week",
        None,
        None,
    ],
})

settings = Settings(backend_type="memory")

async with Medha("batch_df", embedder=embedder, settings=settings) as medha:
    stored = await medha.warm_from_dataframe(
        df,
        question_col="question",
        query_col="generated_query",
        response_col="response_summary",
    )
    print(f"Stored {stored} entries from DataFrame")

    hit = await medha.search("What is the conversion rate?")
    print(f"Search: {hit.strategy.value} — {hit.generated_query}")

## Cell 7: `export_to_dataframe()` — Dump Cache to pandas

`export_to_dataframe()` scrolls the entire collection and returns one row
per `CacheEntry`. Useful for inspection, auditing, or reseeding a new backend.

In [ ]:
import pandas as pd

settings = Settings(backend_type="memory")

async with Medha("batch_export", embedder=embedder, settings=settings) as medha:
    await medha.store_many([
        {"question": f"Export question {i}", "generated_query": f"SELECT {i}"}
        for i in range(5)
    ])

    df = await medha.export_to_dataframe()

print(f"Shape: {df.shape}")
print(df[["original_question", "generated_query", "query_hash"]].to_string(index=False))

## Cell 8: `dedup_collection()` — Remove Duplicate Query Hashes

Multiple questions that generate the **same SQL** share the same `query_hash`.
`dedup_collection()` removes the duplicates, keeping either the newest or
oldest entry depending on `strategy`.

In [ ]:
settings = Settings(backend_type="memory")

async with Medha("batch_dedup", embedder=embedder, settings=settings) as medha:
    # These three questions all map to the same SQL → same query_hash
    await medha.store("Count users",       "SELECT COUNT(*) FROM users")
    await medha.store("How many users?",   "SELECT COUNT(*) FROM users")
    await medha.store("Total user count",  "SELECT COUNT(*) FROM users")
    # Different SQL — should survive deduplication
    await medha.store("List products",     "SELECT * FROM products")

    count_before = await medha._backend.count("batch_dedup")
    print(f"Before dedup: {count_before} entries")

    removed = await medha.dedup_collection(strategy="keep_latest")
    print(f"Removed {removed} duplicates")

    count_after = await medha._backend.count("batch_dedup")
    print(f"After dedup : {count_after} entries")
    assert count_after == count_before - removed

## Pattern: Full Cache Migration

Export from the old backend, re-seed the new one:

```python
# 1. Export from old backend
async with Medha("prod_cache", embedder=embedder, settings=old_settings) as src:
    df = await src.export_to_dataframe()

# 2. Remove duplicates before migrating
df = df.drop_duplicates(subset=["query_hash"], keep="last")

# 3. Seed the new backend
async with Medha("prod_cache", embedder=embedder, settings=new_settings) as dst:
    stored = await dst.warm_from_dataframe(
        df,
        question_col="original_question",
        query_col="generated_query",
        response_col="response_summary",
        batch_size=200,
        on_progress=lambda d, t: print(f"{d}/{t}"),
    )
    print(f"Migration complete: {stored} entries")
```